In [3]:
import pandas as pd

In [4]:
order=pd.read_csv("C:/Users/Durgeshwarnath/OneDrive/Desktop/blinkit/blinkit_orders.csv")
items=pd.read_csv("C:/Users/Durgeshwarnath/OneDrive/Desktop/blinkit/blinkit_order_items.csv")
products=pd.read_csv("C:/Users/Durgeshwarnath/OneDrive/Desktop/blinkit/blinkit_products.csv")
customers=pd.read_csv("C:/Users/Durgeshwarnath/OneDrive/Desktop/blinkit/blinkit_customers.csv")
delivery=pd.read_csv("C:/Users/Durgeshwarnath/OneDrive/Desktop/blinkit/blinkit_delivery_performance.csv")
feedback=pd.read_csv("C:/Users/Durgeshwarnath/OneDrive/Desktop/blinkit/blinkit_customer_feedback.csv")

In [5]:
df= order

In [6]:
df.describe()

,order_id,customer_id,order_total,delivery_partner_id,store_id
count,5.000000e+03,5.000000e+03,5000.00000,5000.000000,5000.000000
mean,5.029129e+09,5.009685e+07,2201.86170,50050.318200,4999.689000
std,2.863533e+09,2.919082e+07,1303.02438,28802.276922,2886.089242
min,6.046500e+04,3.181300e+04,13.25000,43.000000,1.000000
25%,2.531421e+09,2.404314e+07,1086.21500,24928.500000,2509.250000
50%,5.074378e+09,4.997808e+07,2100.69000,50262.500000,4987.000000
75%,7.488579e+09,7.621215e+07,3156.88250,74478.250000,7500.750000
max,9.998298e+09,9.989390e+07,6721.46000,99968.000000,9995.000000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   order_id                5000 non-null   int64  
 1   customer_id             5000 non-null   int64  
 2   order_date              5000 non-null   object 
 3   promised_delivery_time  5000 non-null   object 
 4   actual_delivery_time    5000 non-null   object 
 5   delivery_status         5000 non-null   object 
 6   order_total             5000 non-null   float64
 7   payment_method          5000 non-null   object 
 8   delivery_partner_id     5000 non-null   int64  
 9   store_id                5000 non-null   int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 390.8+ KB


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.isnull().sum()

order_id                  0
customer_id               0
order_date                0
promised_delivery_time    0
actual_delivery_time      0
delivery_status           0
order_total               0
payment_method            0
delivery_partner_id       0
store_id                  0
dtype: int64

In [10]:
df=items

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df.isnull().sum()

order_id      0
product_id    0
quantity      0
unit_price    0
dtype: int64

In [44]:
order_items = order.merge(items,on="order_id",how="left")


In [45]:
print(items.columns)

Index(['order_id', 'product_id', 'quantity', 'unit_price'], dtype='object')


In [46]:
print(products.columns)

Index(['product_id', 'product_name', 'category', 'brand', 'price', 'mrp',
       'margin_percentage', 'shelf_life_days', 'min_stock_level',
       'max_stock_level'],
      dtype='object')


In [47]:
order_items = order_items.merge(products,on="product_id", how="left")

In [48]:
order_items = order_items.merge(customers, on="customer_id", how="left")

In [49]:
order_items = order_items.merge(delivery, on="order_id", how="left")

In [50]:
order_items = order_items.merge(feedback, on= "customer_id", how="left")

In [51]:
print(order_items.columns)

Index(['order_id_x', 'customer_id', 'order_date', 'promised_delivery_time',
       'actual_delivery_time', 'delivery_status_x', 'order_total',
       'payment_method', 'delivery_partner_id_x', 'store_id', 'product_id',
       'quantity', 'unit_price', 'product_name', 'category', 'brand', 'price',
       'mrp', 'margin_percentage', 'shelf_life_days', 'min_stock_level',
       'max_stock_level', 'customer_name', 'email', 'phone', 'address', 'area',
       'pincode', 'registration_date', 'customer_segment', 'total_orders',
       'avg_order_value', 'delivery_partner_id_y', 'promised_time',
       'actual_time', 'delivery_time_minutes', 'distance_km',
       'delivery_status_y', 'reasons_if_delayed', 'feedback_id', 'order_id_y',
       'rating', 'feedback_text', 'feedback_category', 'sentiment',
       'feedback_date'],
      dtype='object')


In [52]:
order_items["revenue"]=order_items["price"]*order_items["quantity"]

In [53]:
print(order_items["revenue"].sum())

14821723.5


In [54]:
order_items["discount"]= order_items["mrp"]-order_items["price"]

In [55]:
order_items["profit"]=order_items["revenue"]-order_items["discount"]

In [56]:
print(order_items["profit"].sum())

11936837.52


In [57]:
order_items["actual_delivery_time"]=pd.to_datetime(order_items["actual_delivery_time"])
order_items["promised_delivery_time"]=pd.to_datetime(order_items["promised_delivery_time"])

In [58]:
order_items["delivery_delay"]= (order_items["actual_delivery_time"] - order_items["promised_delivery_time"]).dt.total_seconds() / 60


In [59]:
order_items["delivery_on_time"]=order_items["delivery_delay"].apply(lambda x:1 if x<=0 else 0)
order_items["order_delayed"]=order_items["delivery_delay"].apply(lambda x:1 if x>0 else 0)

In [60]:
print(order_items["delivery_delay"])

0       -5.0
1       -5.0
2       -5.0
3        2.0
4        2.0
        ... 
14881    1.0
14882    1.0
14883    1.0
14884    1.0
14885    1.0
Name: delivery_delay, Length: 14886, dtype: float64


In [61]:
order_items["order_date"]= pd.to_datetime(order_items["order_date"])


In [62]:
order_items["months"]= order_items["order_date"].dt.month_name()

In [63]:
order_items["weekdays"]= order_items["order_date"].dt.day_name()

In [64]:
order_items["hours"]= order_items["order_date"].dt.hour

In [65]:
order_items["discount_percentage"] = ((order_items["mrp"] - order_items["price"]) / order_items["mrp"]) * 100
order_items["stock_utilization"] = order_items["quantity"] / order_items["max_stock_level"]

In [66]:
pip install openpyxl


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [67]:
order_items.to_excel("blinkit_cleaned.xlsx", index=False)

In [68]:
print(order_items)

       order_id_x  customer_id          order_date promised_delivery_time  \
0      1961864118     30065862 2024-07-17 08:34:01    2024-07-17 08:52:01   
1      1961864118     30065862 2024-07-17 08:34:01    2024-07-17 08:52:01   
2      1961864118     30065862 2024-07-17 08:34:01    2024-07-17 08:52:01   
3      1549769649      9573071 2024-05-28 13:14:29    2024-05-28 13:25:29   
4      1549769649      9573071 2024-05-28 13:14:29    2024-05-28 13:25:29   
...           ...          ...                 ...                    ...   
14881  2494813730     28663279 2023-08-23 12:04:18    2023-08-23 12:20:18   
14882  2494813730     28663279 2023-08-23 12:04:18    2023-08-23 12:20:18   
14883  2494813730     28663279 2023-08-23 12:04:18    2023-08-23 12:20:18   
14884  2494813730     28663279 2023-08-23 12:04:18    2023-08-23 12:20:18   
14885  2494813730     28663279 2023-08-23 12:04:18    2023-08-23 12:20:18   

      actual_delivery_time delivery_status_x  order_total payment_method  \